# 6) Finding optimum conditions via gridsearch
- This code finds region-specific optimum conditions via gridsearch.

# 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import math, pickle
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import seaborn as sns
import glob
import os, sys, gc
from pathlib import Path
import ast
from scipy.signal import find_peaks
from typing import Dict, List, Tuple

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
from utils import eval_util_module 
import importlib
importlib.reload(eval_util_module)

In [ ]:
from utils import utils 
importlib.reload(utils)

In [ ]:
## some params:
param = {'random':[40,41,42,43],
         'control':['lon','lat','month_cos','month_sin','lac_dim']}

feat_var = ['tmin','tmax_ssrd','rh_am','ag_wind_2m']
sub_cols = param['control'] + feat_var
target = 'herd_milk_resid'

# 2. Import Data:

In [ ]:
## ale_df
ale_df = pd.read_csv('3_output/5_1_weighted_ale.csv')

In [ ]:
## full_data.
df = pd.read_parquet('3_output/5_final_herd_detrend_df_full_cow_weight.gzip')

In [ ]:
## model
model = xgb.Booster()
model.load_model(f'3_output/5_xbg_model_full.json')
model.set_param({"device": "cpu"})
model.set_param({'n_jobs':-1})

# 3. Gridsearch

## 3-1. Candidates
- Gridsearch is computationally expensive with large dataset, so we limited our grid search area to coombinations of each weather variable's first to second highest milk producing conditions (see details in SI):
    - candidate ranges were restricted to the 5th to 95th percentile then selected 4 to 5 continuous bins surrounding the peak ALE value; 
    - When two peaks produced similarly high yields, both ranges were retained;
    - if a peak was isolated and surrounded by negative bins, it was excluded as likely noise

In [ ]:
# Parameters
n_bins_around_peak = 4
similarity_threshold = 0.85  # for "similarly high yields"

In [ ]:
all_results = []
# Process each feature one by one
for feat in feat_var:
    print(f"\n{'='*80}")
    print(f"PROCESSING FEATURE: {feat}")
    print('='*80)
    
    for div in ['warm', 'cool']:
        print(f"\n{'-'*60}")
        print(f"Division: {div}")
        print('-'*60)
        
        # Step 1: Get and filter data
        feature_data = ale_df[
            (ale_df['feat_abv'] == feat) & 
            (ale_df['div'] == div)
        ].copy()
        
        if len(feature_data) == 0:
            print(f"  No data found")
            continue
        
        feature_data = feature_data.sort_values('values').reset_index(drop=True)
        
        if len(feature_data) <= 8:
            print(f"  Not enough data points ({len(feature_data)})")
            continue
        
        # Filter to 5th-95th percentile [3:-4]
        filtered_data = feature_data.iloc[3:-4].copy()
        filtered_data = filtered_data.reset_index(drop=True)
        
        print(f"  Bins after filtering: {len(filtered_data)}")
        
        # Step 2: Find positive peaks
        ale_values = filtered_data['ale'].values
        peaks, _ = find_peaks(ale_values, prominence=np.std(ale_values) * 0.1)
        positive_peaks = [p for p in peaks if ale_values[p] > 0]
        
        print(f"  Positive peaks found: {len(positive_peaks)}")
        
        if len(positive_peaks) == 0:
            print(f"  No positive peaks found")
            continue
        
        # Step 3: Filter isolated/noisy peaks
        valid_peaks = []
        for peak_idx in positive_peaks:
            if not utils.is_isolated_or_noisy_peak(ale_values, peak_idx, 
                                            window=2, 
                                            negative_threshold=0.5, 
                                            neighbor_diff_threshold=1.6):
                valid_peaks.append(peak_idx)
        
        print(f"  Valid peaks: {len(valid_peaks)}")
        
        if len(valid_peaks) == 0:
            print(f"  All peaks are isolated/noisy")
            continue
        
        # Step 4: Sort peaks by ALE (highest first)
        peak_info_list = []
        for peak_idx in valid_peaks:
            peak_info_list.append({
                'index': peak_idx,
                'value': filtered_data.iloc[peak_idx]['values'],
                'ale': ale_values[peak_idx]
            })
        peak_info_list = sorted(peak_info_list, key=lambda x: x['ale'], reverse=True)
        
        print(f"\n  Peaks sorted by ALE:")
        for i, p in enumerate(peak_info_list[:5]):  # Show top 5
            print(f"    Peak {i+1}: value={p['value']:.3f}, ALE={p['ale']:.6f}")
        
        # Step 5: Build optimal ranges
        optimal_ranges = utils.build_optimal_ranges(ale_values, filtered_data, peak_info_list, 
                                              similarity_threshold=0.8)
        
        print(f"\n  Optimal ranges created: {len(optimal_ranges)}")
        
        # Visualize
        if len(optimal_ranges) > 0:
            plt.figure(figsize=(14, 6))
            plt.plot(filtered_data['values'], filtered_data['ale'], 'b-', linewidth=2, label='ALE')
            
            # Colors for different ranges
            range_colors = ['green', 'orange', 'purple']
            peak_colors = ['red', 'darkred', 'maroon']
            
            # Plot each range
            for range_idx, range_info in enumerate(optimal_ranges):
                # Highlight this range
                range_data = filtered_data[
                    (filtered_data['values'] >= range_info['min_value']) &
                    (filtered_data['values'] <= range_info['max_value'])
                ]
                
                plt.fill_between(
                    range_data['values'], 
                    0, 
                    range_data['ale'],
                    alpha=0.3,
                    color=range_colors[range_idx % len(range_colors)],
                    label=f"Range {range_idx+1}: [{range_info['min_value']:.2f}, {range_info['max_value']:.2f}] ({range_info['n_peaks']} peak(s))"
                )
                
                # Mark peaks in this range
                for i, (peak_val, peak_ale) in enumerate(zip(range_info['peak_values'], range_info['peak_ales'])):
                    plt.plot(peak_val, peak_ale, 
                            'o', color=peak_colors[range_idx % len(peak_colors)], 
                            markersize=10, 
                            markeredgecolor='black', markeredgewidth=2)
            
            plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
            plt.xlabel(f'{feat}')
            plt.ylabel('ALE')
            plt.title(f'{feat} - {div}: Optimal Ranges with Hierarchical Peak Inclusion')
            plt.legend(loc='best', fontsize=9)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        
        # Step 6: Store results (only essential columns)
        for range_id, range_info in enumerate(optimal_ranges, start=1):
            result = {
                'div': div,
                'feat': feat,
                'range_id': range_id,
                'min_value': range_info['min_value'],
                'max_value': range_info['max_value']
            }
            all_results.append(result)
            
            print(f"Range {range_id}: [{result['min_value']:.3f}, {result['max_value']:.3f}]")

# %%
# Convert to DataFrame
results_df = pd.DataFrame(all_results)

## 3-2. Conditioning:


In [ ]:
warm_result = results_df.loc[results_df['div'] == 'warm'].copy()
cool_result = results_df.loc[results_df['div'] == 'cool'].copy()

# Extract ranges for warm
tmin_warm = warm_result.loc[warm_result['feat'] == 'tmin'].reset_index(drop=True)
tmax_ssrd_warm = warm_result.loc[warm_result['feat'] == 'tmax_ssrd'].reset_index(drop=True)
rh_am_warm = warm_result.loc[warm_result['feat'] == 'rh_am'].reset_index(drop=True)
ag_wind_2m_warm = warm_result.loc[warm_result['feat'] == 'ag_wind_2m'].reset_index(drop=True)

temp_warm = df.loc[(
    (df['div'] == 'warm') &
    (((df['tmin'] >= tmin_warm.iloc[0]['min_value']) & 
      (df['tmin'] <= tmin_warm.iloc[0]['max_value'])) |
     ((df['tmin'] >= tmin_warm.iloc[1]['min_value']) & 
      (df['tmin'] <= tmin_warm.iloc[1]['max_value']))) &
    
    (((df['tmax_ssrd'] >= tmax_ssrd_warm.iloc[0]['min_value']) & 
      (df['tmax_ssrd'] <= tmax_ssrd_warm.iloc[0]['max_value'])) |
     ((df['tmax_ssrd'] >= tmax_ssrd_warm.iloc[1]['min_value']) & 
      (df['tmax_ssrd'] <= tmax_ssrd_warm.iloc[1]['max_value']))) &
    
    (((df['rh_am'] >= rh_am_warm.iloc[0]['min_value']) & 
      (df['rh_am'] <= rh_am_warm.iloc[0]['max_value'])) |
     ((df['rh_am'] >= rh_am_warm.iloc[1]['min_value']) & 
      (df['rh_am'] <= rh_am_warm.iloc[1]['max_value']))) &
    
    (((df['ag_wind_2m'] >= ag_wind_2m_warm.iloc[0]['min_value']) & 
      (df['ag_wind_2m'] <= ag_wind_2m_warm.iloc[0]['max_value'])) |
     ((df['ag_wind_2m'] >= ag_wind_2m_warm.iloc[1]['min_value']) & 
      (df['ag_wind_2m'] <= ag_wind_2m_warm.iloc[1]['max_value'])))
)].copy()

print(f"Warm division: {len(temp_warm)} rows")

# Do the same for cool
tmin_cool = cool_result.loc[cool_result['feat'] == 'tmin'].reset_index(drop=True)
tmax_ssrd_cool = cool_result.loc[cool_result['feat'] == 'tmax_ssrd'].reset_index(drop=True)
rh_am_cool = cool_result.loc[cool_result['feat'] == 'rh_am'].reset_index(drop=True)
ag_wind_2m_cool = cool_result.loc[cool_result['feat'] == 'ag_wind_2m'].reset_index(drop=True)

temp_cool = df.loc[(
    (df['div'] == 'cool') &
    (((df['tmin'] >= tmin_cool.iloc[0]['min_value']) & 
      (df['tmin'] <= tmin_cool.iloc[0]['max_value']))) &
    
    (((df['tmax_ssrd'] >= tmax_ssrd_cool.iloc[0]['min_value']) & 
      (df['tmax_ssrd'] <= tmax_ssrd_cool.iloc[0]['max_value'])) |
     ((df['tmax_ssrd'] >= tmax_ssrd_cool.iloc[1]['min_value']) & 
      (df['tmax_ssrd'] <= tmax_ssrd_cool.iloc[1]['max_value']))) &
    
    (((df['rh_am'] >= rh_am_cool.iloc[0]['min_value']) & 
      (df['rh_am'] <= rh_am_cool.iloc[0]['max_value'])) |
     ((df['rh_am'] >= rh_am_cool.iloc[1]['min_value']) & 
      (df['rh_am'] <= rh_am_cool.iloc[1]['max_value']))) &
    
    (((df['ag_wind_2m'] >= ag_wind_2m_cool.iloc[0]['min_value']) & 
      (df['ag_wind_2m'] <= ag_wind_2m_cool.iloc[0]['max_value'])) |
     ((df['ag_wind_2m'] >= ag_wind_2m_cool.iloc[1]['min_value']) & 
      (df['ag_wind_2m'] <= ag_wind_2m_cool.iloc[1]['max_value'])))
)].copy()

print(f"Cool division: {len(temp_cool)} rows")

# Combine
opt_candidates = pd.concat([temp_warm, temp_cool], axis=0, ignore_index=True)
print(f"\nTotal optimal candidates: {len(opt_candidates)}")

## 3-3. Gridsearch

In [ ]:

for div in ['warm','cool']:
    print('-'*5, div)
    temp = opt_candidates.loc[opt_candidates['div'] == div].copy()
    
    df_temp = df.copy()
    
    print(temp.shape[0])
    
    for i in temp.index:
        #print(i)
        for var in feat_var:
            df_temp.loc[df_temp['div'] == div, var] = temp.loc[i,var]
            
        train = xgb.DMatrix(df_temp[sub_cols])
        pyield = model.predict(train)
        del train
        gc.collect()
        opt_candidates.loc[i, 'pyield'] = np.mean(pyield)
        opt_candidates.loc[i, 'wpyield'] = np.average(pyield, weights= df_temp['final_weight'])
    
    del temp, df_temp            

In [ ]:
opt_candidates.loc[opt_candidates['div'] == 'warm'].sort_values(by='wpyield', ascending=False)[['div','state_abv','wpyield','pyield','GEOID','date']+sub_cols].iloc[0,:]

In [ ]:
opt_candidates.loc[opt_candidates['div'] == 'cool'].sort_values(by='wpyield', ascending=False)[['div','state_abv','wpyield','pyield','GEOID','date']+sub_cols].iloc[0,:]

In [ ]:
opt_candidates.to_csv('3_output/6_grid_search_output.csv')